# Лабораторна робота №4: дерева рішень та ансамблі — якість, стійкість, інтерпретація

**Варіант:** 7 — Titanic (набір і split з ЛР2, ціль `AtRisk = 1 − Survived`).
**Головне питання:** чи виправданий перехід від окремого дерева рішень до ансамблю з урахуванням якості, стійкості та можливості пояснення?

In [1]:
import json
from pathlib import Path
import pickle
import platform
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.inspection import PartialDependenceDisplay, permutation_importance
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, confusion_matrix, f1_score
from sklearn.model_selection import (GridSearchCV, RepeatedStratifiedKFold, StratifiedKFold, cross_val_predict,
                                     cross_validate)
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier, plot_tree

PROJECT_ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(PROJECT_ROOT))

from src.titanic_workflow import (CATEGORICAL_FEATURES, FEATURES, NUMERIC_FEATURES, RANDOM_STATE, load_titanic,
                                  make_cv, make_preprocessor, split_features_target, stratified_split)

WORK_DIR = Path.cwd().parent
FIGURES = WORK_DIR / "figures"
RESULTS = WORK_DIR / "results"
CONFIGS = WORK_DIR / "configs"
for folder in (FIGURES, RESULTS, CONFIGS):
    folder.mkdir(exist_ok=True)
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 30)

## 1. Дослідницьке питання, гіпотези та протокол

| № | Гіпотеза | Показник | Правило висновку |
|---|---|---|---|
| H1 | Ансамбль дасть вищий середній macro F1, ніж контрольоване дерево | CV macro F1 на однакових 20 зовнішніх fold-ах (дерево — nested CV) | підтверджено, якщо різниця > SD дерева |
| H2 | Прогнози випадкового лісу менш чутливі до зміни fold-ів | частка відхилень OOF-прогнозу від консенсусу за 10 seed-ами | у RF менше, ніж у дерева, щонайменше вдвічі |
| H3 | Обмеження глибини зменшить розрив train − CV | gap для сітки структури | gap монотонно зростає з глибиною |
| H4 | Ранг важливості ознак змінюється між fold-ами | permutation importance по fold-ах, частота в top-3 | хоча б одна ознака з top-3 входить у нього < 100% fold-ів |

**Протокол (зафіксовано до навчання):**

| Елемент | Значення |
|---|---|
| Дані | `data/titanic.csv` (Kaggle `train.csv`, 891 × 12), ціль `AtRisk` |
| Метрика | **macro F1** — позитивний клас є більшістю, F1 класу 1 завищує baseline (ЛР2); додатково Accuracy |
| Split | стратифікований 80/20, `random_state=42` (712 / 179), як у ЛР2–ЛР3, ПР4 |
| Зовнішня CV | `RepeatedStratifiedKFold(5 × 4, random_state=42)` = 20 fold-ів, однакові для всіх моделей |
| Дерево | nested CV: внутрішній `StratifiedKFold(4)` обирає `max_depth ∈ {2,3,4,5,6,8,None}` × `min_samples_leaf ∈ {1,2,5,10,20}` за **правилом однієї SE**: серед конфігурацій із CV ≥ max − SE обирається найменша глибина, далі найбільший `min_samples_leaf` |
| Ансамблі | RF: 300 дерев, `max_features="sqrt"`, `min_samples_leaf=2`; HGB: 200 ітерацій, `learning_rate=0.05`, `max_leaf_nodes=15`, `l2_regularization=1.0` |
| Preprocessing | медіана + індикатор пропуску для числових, мода + one-hot для категоріальних; масштабування не застосовується (дерева інваріантні до монотонних перетворень) |
| Правило вибору | ансамбль обирається, лише якщо його CV macro F1 вищий за дерево більш ніж на SD дерева **і** прогнози стабільніші; інакше — контрольоване дерево (простіше пояснити) |

**Задокументована зміна протоколу.** У першому запуску структура дерева обиралась за простим максимумом CV macro F1 і дала `max_depth=8, min_samples_leaf=1` (60 листків, CV 0,803 ± 0,041). Таке дерево не читається людиною, а його перевага над деревом глибини 3 (8 листків, 0,791 ± 0,021) менша за стандартну похибку. Тому, як і в ПР3, для вибору структури застосовано правило однієї SE; рішення прийнято до перегляду test, ансамблі не змінювались. Результати першого запуску збережено в `results/tree_structure_search.csv` (повна сітка).

## 2. Дані та фіксований поділ

In [2]:
df = load_titanic()
X, y = split_features_target(df)
X_train, X_test, y_train, y_test = stratified_split(X, y)
check = pd.DataFrame({
    "Показник": ["Об'єктів", "Ознак (змістовних)", "Класів", "Клас 0 (вижив)", "Клас 1 (не вижив)", "Пропуски Age", "Пропуски Embarked",
                 "Повні дублікати рядків", "Константні ознаки"],
    "Значення": [len(X), X.shape[1], y.nunique(), f"{(y == 0).sum()} ({(y == 0).mean():.1%})", f"{(y == 1).sum()} ({(y == 1).mean():.1%})",
                 X["Age"].isna().sum(), X["Embarked"].isna().sum(), df.duplicated().sum(), int((X.nunique() <= 1).sum())],
})
display(check)
print("Ризики витоку: Survived вилучено; PassengerId/Name/Ticket не використовуються; Cabin (77% пропусків) не використовується.")
print("Кореляція ознак з ціллю не перевищує:", X.select_dtypes("number").corrwith(y).abs().max().round(3))
pd.DataFrame({"PassengerId": X_train.index, "part": "train"}).pipe(
    lambda d: pd.concat([d, pd.DataFrame({"PassengerId": X_test.index, "part": "test"})])).to_csv(RESULTS / "split_ids.csv", index=False)
print("Train / test:", len(X_train), "/", len(X_test))


def tree_pipeline(model):
    return Pipeline([("preprocessor", make_preprocessor(scale=None)), ("model", model)])

,Показник,Значення
0,Об'єктів,891
1,Ознак (змістовних),7
2,Класів,2
3,Клас 0 (вижив),342 (38.4%)
4,Клас 1 (не вижив),549 (61.6%)
5,Пропуски Age,177
6,Пропуски Embarked,2
7,Повні дублікати рядків,0
8,Константні ознаки,0


Ризики витоку: Survived вилучено; PassengerId/Name/Ticket не використовуються; Cabin (77% пропусків) не використовується.
Кореляція ознак з ціллю не перевищує: 0.338
Train / test: 712 / 179


## 3. Контрольна точка та окреме дерево

In [3]:
inner_cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=17)


def one_se_refit(cv_results):
    """Найпростіше дерево серед конфігурацій із CV ≥ max − SE (SE = std / √k)."""
    res = pd.DataFrame(cv_results)
    n_splits = sum(c.startswith("split") and c.endswith("_test_score") for c in res.columns)
    best = res["mean_test_score"].idxmax()
    threshold = res.loc[best, "mean_test_score"] - res.loc[best, "std_test_score"] / np.sqrt(n_splits)
    ok = res[res["mean_test_score"] >= threshold].copy()
    ok["depth_key"] = ok["param_model__max_depth"].apply(lambda d: 1_000 if d is None or pd.isna(d) else d)
    return int(ok.sort_values(["depth_key", "param_model__min_samples_leaf"], ascending=[True, False]).index[0])


TREE_GRID = {"model__max_depth": [2, 3, 4, 5, 6, 8, None], "model__min_samples_leaf": [1, 2, 5, 10, 20]}
structure_cv = make_cv()
tree_search = GridSearchCV(tree_pipeline(DecisionTreeClassifier(random_state=RANDOM_STATE)), TREE_GRID, scoring="f1_macro",
                           cv=structure_cv, return_train_score=True, refit=one_se_refit, n_jobs=2).fit(X_train, y_train)
res = pd.DataFrame(tree_search.cv_results_)
rows = []
for i, params in enumerate(res["params"]):
    fitted = tree_pipeline(DecisionTreeClassifier(random_state=RANDOM_STATE, max_depth=params["model__max_depth"],
                                                  min_samples_leaf=params["model__min_samples_leaf"])).fit(X_train, y_train)
    tree = fitted.named_steps["model"]
    rows.append({"max_depth": "None" if params["model__max_depth"] is None else params["model__max_depth"],
                 "min_samples_leaf": params["model__min_samples_leaf"], "Train F1": res.loc[i, "mean_train_score"],
                 "CV F1": res.loc[i, "mean_test_score"], "CV std": res.loc[i, "std_test_score"],
                 "Gap": res.loc[i, "mean_train_score"] - res.loc[i, "mean_test_score"], "Глибина": tree.get_depth(),
                 "Листків": tree.get_n_leaves(), "Вузлів": tree.tree_.node_count})
structure = pd.DataFrame(rows)
structure.to_csv(RESULTS / "tree_structure_search.csv", index=False)
display(structure.sort_values("CV F1", ascending=False).head(8).style.format(
    {"Train F1": "{:.3f}", "CV F1": "{:.3f}", "CV std": "{:.3f}", "Gap": "{:.3f}"}))
best_raw = res["mean_test_score"].idxmax()
print("Максимум CV:", res.loc[best_raw, "params"], f"{res.loc[best_raw, 'mean_test_score']:.3f} ± {res.loc[best_raw, 'std_test_score']:.3f}",
      f"| поріг 1-SE = {res.loc[best_raw, 'mean_test_score'] - res.loc[best_raw, 'std_test_score'] / np.sqrt(5):.3f}")
print("Обрано за 1-SE:", tree_search.best_params_,
      f"CV macro F1 = {res.loc[tree_search.best_index_, 'mean_test_score']:.3f} ± {res.loc[tree_search.best_index_, 'std_test_score']:.3f}")
unlimited = structure[(structure["max_depth"] == "None") & (structure["min_samples_leaf"] == 1)].iloc[0]
print(f"Необмежене дерево: train {unlimited['Train F1']:.3f}, CV {unlimited['CV F1']:.3f}, глибина {unlimited['Глибина']}, "
      f"листків {unlimited['Листків']}")

,max_depth,min_samples_leaf,Train F1,CV F1,CV std,Gap,Глибина,Листків,Вузлів
25,8,1,0.898,0.803,0.041,0.095,8,60,119
27,8,5,0.869,0.801,0.016,0.068,8,45,89
22,6,5,0.854,0.798,0.014,0.057,6,29,57
21,6,2,0.858,0.793,0.016,0.065,6,30,59
5,3,1,0.817,0.791,0.021,0.025,3,8,15
32,None,5,0.874,0.790,0.023,0.084,16,73,145
20,6,1,0.866,0.790,0.021,0.076,6,32,63
6,3,2,0.816,0.787,0.018,0.029,3,8,15


Максимум CV: {'model__max_depth': 8, 'model__min_samples_leaf': 1} 0.803 ± 0.041 | поріг 1-SE = 0.785
Обрано за 1-SE: {'model__max_depth': 3, 'model__min_samples_leaf': 2} CV macro F1 = 0.787 ± 0.018
Необмежене дерево: train 0.984, CV 0.764, глибина 21, листків 165


In [4]:
order = ["2", "3", "4", "5", "6", "8", "None"]
structure["max_depth_s"] = structure["max_depth"].astype(str)
fig, axes = plt.subplots(1, 3, figsize=(18, 4.8))
sns.heatmap(structure.pivot(index="min_samples_leaf", columns="max_depth_s", values="CV F1")[order], annot=True, fmt=".3f",
            cmap="viridis", ax=axes[0], cbar_kws={"label": "CV macro F1"})
axes[0].set(title="CV macro F1 дерева", xlabel="max_depth", ylabel="min_samples_leaf")
sns.heatmap(structure.pivot(index="min_samples_leaf", columns="max_depth_s", values="Gap")[order], annot=True, fmt=".3f",
            cmap="Reds", ax=axes[1], cbar_kws={"label": "train − CV"})
axes[1].set(title="Розрив train − CV", xlabel="max_depth", ylabel="min_samples_leaf")
for leaf, color in zip([1, 5, 20], ["#c46a5a", "#3f7f93", "#6a8f3f"]):
    part = structure[structure["min_samples_leaf"] == leaf]
    axes[2].plot(part["Листків"], part["Train F1"], ls="--", marker="o", color=color, alpha=0.7)
    axes[2].plot(part["Листків"], part["CV F1"], marker="o", color=color, label=f"min_samples_leaf={leaf} (суцільна — CV, пунктир — train)")
axes[2].set(xscale="log", title="Якість від кількості листків", xlabel="Листків (log)", ylabel="macro F1")
axes[2].legend(fontsize=7)
fig.tight_layout()
fig.savefig(FIGURES / "tree_structure.png", dpi=160, bbox_inches="tight")
plt.show()

In [5]:
best_tree = tree_search.best_estimator_
tree_model = best_tree.named_steps["model"]
feature_names = list(best_tree.named_steps["preprocessor"].get_feature_names_out())
print(f"Контрольоване дерево: глибина {tree_model.get_depth()}, листків {tree_model.get_n_leaves()}, вузлів {tree_model.tree_.node_count}")
fig, ax = plt.subplots(figsize=(22, 9))
plot_tree(tree_model, feature_names=feature_names, class_names=["вижив", "не вижив"], filled=True, rounded=True,
          max_depth=3, fontsize=8, proportion=True, impurity=True, ax=ax)
ax.set_title("Контрольоване дерево (у вузлі — умова, gini, частка об'єктів, розподіл класів [вижив, не вижив])")
fig.tight_layout()
fig.savefig(FIGURES / "controlled_tree.png", dpi=140, bbox_inches="tight")
plt.show()

Контрольоване дерево: глибина 3, листків 8, вузлів 15


**Шлях конкретного об'єкта.** Для локального пояснення беремо навчального пасажира, якого контрольоване дерево класифікує неправильно, і відтворюємо всі умовні переходи.

In [6]:
Z_train = best_tree.named_steps["preprocessor"].transform(X_train)
train_pred = tree_model.predict(Z_train)
wrong = np.flatnonzero(train_pred != y_train.values)
row_id = wrong[0]
pid = X_train.index[row_id]
node_indicator = tree_model.decision_path(Z_train[row_id:row_id + 1])
nodes = node_indicator.indices[node_indicator.indptr[0]:node_indicator.indptr[1]]
leaf = tree_model.apply(Z_train[row_id:row_id + 1])[0]
path_rows = []
for node in nodes:
    if node == leaf:
        dist = tree_model.tree_.value[node][0]
        path_rows.append({"Вузол": node, "Ознака": "— листок —", "Поріг": np.nan, "Значення об'єкта": np.nan, "Перехід": "",
                          "Розподіл у вузлі [вижив, не вижив]": np.round(dist / dist.sum(), 3)})
        continue
    f = tree_model.tree_.feature[node]
    thr = tree_model.tree_.threshold[node]
    value = Z_train[row_id, f]
    path_rows.append({"Вузол": node, "Ознака": feature_names[f], "Поріг": thr, "Значення об'єкта": value,
                      "Перехід": "≤ → ліворуч" if value <= thr else "> → праворуч", "Розподіл у вузлі [вижив, не вижив]": ""})
path = pd.DataFrame(path_rows)
print(f"PassengerId {pid}: {df.set_index('PassengerId').loc[pid, 'Name']}")
display(X_train.loc[[pid]])
print(f"Факт: {y_train.loc[pid]} | прогноз дерева: {train_pred[row_id]}")
display(path.style.format({"Поріг": "{:.3f}", "Значення об'єкта": "{:.3f}"}, na_rep=""))
path.to_csv(RESULTS / "decision_path_example.csv", index=False)

PassengerId 853: Boulos, Miss. Nourelain


,Age,SibSp,Parch,Fare,Pclass,Sex,Embarked
PassengerId,,,,,,,
853,9.0,1,1,15.2458,3,female,C


Факт: 1 | прогноз дерева: 0


,Вузол,Ознака,Поріг,Значення об'єкта,Перехід,"Розподіл у вузлі [вижив, не вижив]"
0,0,Sex_female,0.500,1.000,> → праворуч,
1,8,Pclass_3,0.500,1.000,> → праворуч,
2,12,Fare,23.350,15.246,≤ → ліворуч,
3,13,— листок —,,,,[0.547 0.453]


## 4. Ансамблеві моделі

- **RandomForestClassifier**: `n_estimators=300` (після ~200 дерев усереднення майже не змінюється, а час росте лінійно), `max_features="sqrt"` (≈ 3 з 13 стовпців у вузлі — декорелює дерева), `min_samples_leaf=2` (не дає листкам запам'ятовувати поодиноких пасажирів), `random_state=42`.
- **HistGradientBoostingClassifier**: `max_iter=200` × `learning_rate=0.05` (обережні кроки), `max_leaf_nodes=15` (неглибокі базові дерева), `l2_regularization=1.0`, `early_stopping=False` (однакова кількість ітерацій у кожному fold-і), `random_state=42`.

In [7]:
models = {
    "Baseline": tree_pipeline(DummyClassifier(strategy="most_frequent")),
    "Необмежене дерево": tree_pipeline(DecisionTreeClassifier(random_state=RANDOM_STATE)),
    "Контрольоване дерево (nested)": GridSearchCV(tree_pipeline(DecisionTreeClassifier(random_state=RANDOM_STATE)), TREE_GRID,
                                                   scoring="f1_macro", cv=inner_cv, refit=one_se_refit, n_jobs=1),
    "Random Forest": tree_pipeline(RandomForestClassifier(n_estimators=300, max_features="sqrt", min_samples_leaf=2,
                                                          random_state=RANDOM_STATE, n_jobs=1)),
    "HistGradientBoosting": tree_pipeline(HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, max_leaf_nodes=15,
                                                                         l2_regularization=1.0, early_stopping=False,
                                                                         random_state=RANDOM_STATE)),
}

## 5. Порівняння якості на однакових 20 зовнішніх fold-ах

In [8]:
outer_cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=4, random_state=RANDOM_STATE)
fold_rows, summary_rows = [], []
for name, estimator in models.items():
    scores = cross_validate(estimator, X_train, y_train, cv=outer_cv, scoring={"f1": "f1_macro", "acc": "accuracy"},
                            return_train_score=True, n_jobs=2)
    for i, v in enumerate(scores["test_f1"]):
        fold_rows.append({"Модель": name, "fold": i, "macro F1": v})
    summary_rows.append({"Модель": name, "Train F1": scores["train_f1"].mean(), "CV F1": scores["test_f1"].mean(),
                         "CV std": scores["test_f1"].std(ddof=1), "CV min": scores["test_f1"].min(),
                         "Gap": scores["train_f1"].mean() - scores["test_f1"].mean(), "CV Accuracy": scores["test_acc"].mean()})
quality = pd.DataFrame(summary_rows)
quality_folds = pd.DataFrame(fold_rows)
quality.to_csv(RESULTS / "quality_comparison.csv", index=False)
quality_folds.to_csv(RESULTS / "quality_folds.csv", index=False)
display(quality.style.format({c: "{:.3f}" for c in quality.columns if c != "Модель"}))

pivot = quality_folds.pivot(index="fold", columns="Модель", values="macro F1")
for a, b in [("Random Forest", "Контрольоване дерево (nested)"), ("HistGradientBoosting", "Контрольоване дерево (nested)"),
             ("Random Forest", "HistGradientBoosting")]:
    diff = pivot[a] - pivot[b]
    print(f"{a} − {b}: середня різниця {diff.mean():+.3f}, {a} кращий у {(diff > 0).sum()} з 20 fold-ів")

fig, ax = plt.subplots(figsize=(11, 4.8))
order_models = [m for m in models if m != "Baseline"]
sns.boxplot(data=quality_folds[quality_folds["Модель"] != "Baseline"], x="Модель", y="macro F1", order=order_models, ax=ax,
            color="#d9e6ea", width=0.5)
sns.stripplot(data=quality_folds[quality_folds["Модель"] != "Baseline"], x="Модель", y="macro F1", order=order_models, ax=ax,
              color="#3f7f93", size=4, jitter=0.15)
ax.axhline(quality.set_index("Модель").loc["Baseline", "CV F1"], color="#c46a5a", ls="--", label="Baseline")
ax.set(title="Розподіл macro F1 на 20 спільних зовнішніх fold-ах", xlabel="", ylabel="macro F1")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES / "quality_boxplot.png", dpi=160, bbox_inches="tight")
plt.show()

,Модель,Train F1,CV F1,CV std,CV min,Gap,CV Accuracy
0,Baseline,0.381,0.381,0.001,0.380,0.000,0.617
1,Необмежене дерево,0.984,0.757,0.029,0.707,0.228,0.770
2,Контрольоване дерево (nested),0.809,0.789,0.033,0.717,0.020,0.809
3,Random Forest,0.910,0.804,0.030,0.756,0.106,0.821
4,HistGradientBoosting,0.916,0.801,0.028,0.745,0.115,0.816


Random Forest − Контрольоване дерево (nested): середня різниця +0.014, Random Forest кращий у 14 з 20 fold-ів
HistGradientBoosting − Контрольоване дерево (nested): середня різниця +0.011, HistGradientBoosting кращий у 15 з 20 fold-ів
Random Forest − HistGradientBoosting: середня різниця +0.003, Random Forest кращий у 11 з 20 fold-ів


## 6. Стійкість до зміни fold-ів (10 значень `random_state`)

Гіперпараметри зафіксовано: контрольоване дерево використовує структуру, обрану на кроці 3 (метою є стійкість конфігурації, а не повторна оцінка пошуку).

In [9]:
fixed = {"Контрольоване дерево": clone(best_tree), "Random Forest": models["Random Forest"],
         "HistGradientBoosting": models["HistGradientBoosting"]}
SEEDS = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
stab_models = {**fixed, "Необмежене дерево": models["Необмежене дерево"]}  # необмежене — для контрасту
seed_rows, oof_store = [], {name: [] for name in stab_models}
for seed in SEEDS:
    cv_seed = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    for name, est in stab_models.items():
        pred = cross_val_predict(est, X_train, y_train, cv=cv_seed, n_jobs=2)
        oof_store[name].append(pred)
        seed_rows.append({"Модель": name, "seed": seed, "macro F1": f1_score(y_train, pred, average="macro")})
seeds = pd.DataFrame(seed_rows)
seeds.to_csv(RESULTS / "stability_seeds.csv", index=False)

stab_rows, unstable_objects = [], {}
for name, preds in oof_store.items():
    P = np.vstack(preds)
    consensus = (P.mean(0) >= 0.5).astype(int)
    deviation = (P != consensus).mean(0)
    unstable_objects[name] = deviation
    q = seeds[seeds["Модель"] == name]["macro F1"]
    stab_rows.append({"Модель": name, "F1 mean": q.mean(), "F1 std": q.std(ddof=1), "F1 min": q.min(), "F1 max": q.max(),
                      "Розбіжність з консенсусом": deviation.mean(), "Нестійких об'єктів (> 25%)": int((deviation > 0.25).sum())})
stability = pd.DataFrame(stab_rows)
stability.to_csv(RESULTS / "stability_summary.csv", index=False)
pd.DataFrame(unstable_objects, index=X_train.index).to_csv(RESULTS / "oof_instability_per_object.csv")
display(stability.style.format({"F1 mean": "{:.3f}", "F1 std": "{:.3f}", "F1 min": "{:.3f}", "F1 max": "{:.3f}",
                                "Розбіжність з консенсусом": "{:.2%}"}))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
sns.pointplot(data=seeds, x="Модель", y="macro F1", errorbar=("pi", 100), capsize=0.15, ax=axes[0], color="#3f7f93")
sns.stripplot(data=seeds, x="Модель", y="macro F1", ax=axes[0], color="#c46a5a", size=4, alpha=0.7)
axes[0].set(title="OOF macro F1 за 10 схемами fold-ів (min–max)", xlabel="", ylabel="macro F1")
axes[1].bar(stability["Модель"], stability["Розбіжність з консенсусом"] * 100, color=["#c46a5a", "#3f7f93", "#6a8f3f", "#8a6aa8"])
axes[1].tick_params(axis="x", labelsize=8)
axes[0].tick_params(axis="x", labelsize=8)
for i, (v, n) in enumerate(zip(stability["Розбіжність з консенсусом"], stability["Нестійких об'єктів (> 25%)"])):
    axes[1].text(i, v * 100 + 0.1, f"{v:.2%}\n{n} нестійких", ha="center", fontsize=9)
axes[1].set(title="Частка OOF-прогнозів, що відрізняються від консенсусу", ylabel="%", xlabel="")
fig.tight_layout()
fig.savefig(FIGURES / "stability.png", dpi=160, bbox_inches="tight")
plt.show()

,Модель,F1 mean,F1 std,F1 min,F1 max,Розбіжність з консенсусом,Нестійких об'єктів (> 25%)
0,Контрольоване дерево,0.797,0.010,0.779,0.809,2.23%,16
1,Random Forest,0.803,0.007,0.793,0.817,3.57%,43
2,HistGradientBoosting,0.810,0.009,0.792,0.823,4.47%,54
3,Необмежене дерево,0.760,0.010,0.746,0.778,8.90%,109


## 7. Стійкість структури дерева: 150 bootstrap-вибірок

In [10]:
rng = np.random.default_rng(RANDOM_STATE)
original_of = {name: next((f for f in FEATURES if name == f or name.startswith(f + "_") or name.endswith("_" + f)), name)
               for name in feature_names}
original_of["missingindicator_Age"] = "Age (пропуск)"
boot_rows = []
for b in range(150):
    idx = rng.integers(0, len(X_train), len(X_train))
    fitted = clone(best_tree).fit(X_train.iloc[idx], y_train.iloc[idx])
    t = fitted.named_steps["model"]
    names = list(fitted.named_steps["preprocessor"].get_feature_names_out())
    root = names[t.tree_.feature[0]]
    level1 = sorted(names[t.tree_.feature[c]] for c in (t.tree_.children_left[0], t.tree_.children_right[0]) if t.tree_.feature[c] >= 0)
    boot_rows.append({"b": b, "root_column": root, "root_feature": original_of.get(root, root), "root_threshold": t.tree_.threshold[0],
                      "depth": t.get_depth(), "leaves": t.get_n_leaves(), "level1": " | ".join(level1)})
boot = pd.DataFrame(boot_rows)
boot.to_csv(RESULTS / "bootstrap_tree_structure.csv", index=False)
display(boot["root_column"].value_counts().to_frame("частота кореня"))
display(boot.groupby("root_column")["root_threshold"].describe()[["count", "mean", "min", "max"]].style.format("{:.3f}"))
display(boot[["depth", "leaves"]].describe().loc[["mean", "std", "min", "max"]].style.format("{:.2f}"))
display(boot["level1"].value_counts().head(6).to_frame("частота пари ознак другого рівня"))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
boot["root_column"].value_counts().plot(kind="bar", ax=axes[0], color="#3f7f93", rot=0)
axes[0].set(title="Ознака кореневого вузла у 150 bootstrap-деревах", ylabel="Кількість дерев", xlabel="")
boot["leaves"].plot(kind="hist", bins=range(boot["leaves"].min(), boot["leaves"].max() + 2), ax=axes[1], color="#c46a5a")
axes[1].set(title="Кількість листків bootstrap-дерев (параметри фіксовані)", xlabel="Листків", ylabel="Дерев")
fig.tight_layout()
fig.savefig(FIGURES / "bootstrap_root.png", dpi=160, bbox_inches="tight")
plt.show()

,частота кореня
root_column,
Sex_female,150


,count,mean,min,max
root_column,,,,
Sex_female,150.000,0.500,0.500,0.500


,depth,leaves
mean,3.00,7.92
std,0.00,0.27
min,3.00,7.00
max,3.00,8.00


,частота пари ознак другого рівня
level1,
Age | Pclass_3,113
Pclass_1 | Pclass_3,27
Fare | Pclass_3,10


## 8. Важливість ознак та її варіативність

Impurity-based importance рахується для one-hot стовпців, тому для порівняння сумується до 7 вихідних ознак. Permutation importance рахується на валідаційній частині кожного з 5 fold-ів (`StratifiedKFold(5, random_state=42)`), переставляються **вихідні** стовпці (`Sex`, `Pclass`, …), 20 повторів, метрика macro F1. Оскільки ознак лише 7, крім top-5 (вимога методички) показуємо частоту в top-3 — вона інформативніша.

In [11]:
def impurity_by_feature(pipe):
    names = pipe.named_steps["preprocessor"].get_feature_names_out()
    imp = pd.Series(pipe.named_steps["model"].feature_importances_, index=names)
    grouped = imp.groupby(lambda n: "Age" if n == "missingindicator_Age" else original_of.get(n, n)).sum()
    return grouped.reindex(FEATURES)


tree_full = clone(best_tree).fit(X_train, y_train)
rf_full = clone(models["Random Forest"]).fit(X_train, y_train)
impurity = pd.DataFrame({"Дерево": impurity_by_feature(tree_full), "Random Forest": impurity_by_feature(rf_full)})

perm_rows = []
for fold, (tr, va) in enumerate(make_cv().split(X_train, y_train), 1):
    for name, est in [("Дерево", best_tree), ("Random Forest", models["Random Forest"])]:
        fitted = clone(est).fit(X_train.iloc[tr], y_train.iloc[tr])
        pi = permutation_importance(fitted, X_train.iloc[va], y_train.iloc[va], scoring="f1_macro", n_repeats=20,
                                    random_state=100 + fold, n_jobs=2)
        ranks = pd.Series(pi.importances_mean, index=FEATURES).rank(ascending=False, method="first")
        for f, m, s in zip(FEATURES, pi.importances_mean, pi.importances_std):
            perm_rows.append({"Модель": name, "fold": fold, "Ознака": f, "PI": m, "PI repeat std": s, "rank": ranks[f]})
perm = pd.DataFrame(perm_rows)
perm.to_csv(RESULTS / "permutation_importance_folds.csv", index=False)
perm_summary = perm.groupby(["Модель", "Ознака"]).agg(PI_mean=("PI", "mean"), PI_fold_std=("PI", "std"),
                                                     top3=("rank", lambda r: (r <= 3).mean()), top5=("rank", lambda r: (r <= 5).mean()),
                                                     rank_min=("rank", "min"), rank_max=("rank", "max")).reset_index()
perm_summary["Impurity"] = [impurity.loc[f, m] for m, f in zip(perm_summary["Модель"], perm_summary["Ознака"])]
perm_summary = perm_summary.sort_values(["Модель", "PI_mean"], ascending=[True, False])
perm_summary.to_csv(RESULTS / "importance_summary.csv", index=False)
display(perm_summary.style.format({"PI_mean": "{:.3f}", "PI_fold_std": "{:.3f}", "top3": "{:.0%}", "top5": "{:.0%}",
                                   "rank_min": "{:.0f}", "rank_max": "{:.0f}", "Impurity": "{:.3f}"}))
corr = pd.DataFrame({"Fare": X_train["Fare"], "Pclass": X_train["Pclass"], "Sex_male": (X_train["Sex"] == "male").astype(int),
                     "Age": X_train["Age"], "SibSp": X_train["SibSp"], "Parch": X_train["Parch"]}).corr(method="spearman")
print("Spearman Fare–Pclass:", round(corr.loc["Fare", "Pclass"], 3), "| SibSp–Parch:", round(corr.loc["SibSp", "Parch"], 3),
      "| Fare–Sex_male:", round(corr.loc["Fare", "Sex_male"], 3))

,Модель,Ознака,PI_mean,PI_fold_std,top3,top5,rank_min,rank_max,Impurity
5,Random Forest,Sex,0.206,0.012,100%,100%,1,1,0.366
4,Random Forest,Pclass,0.089,0.022,100%,100%,2,2,0.144
0,Random Forest,Age,0.049,0.023,100%,100%,3,3,0.186
1,Random Forest,Embarked,0.011,0.013,0%,40%,4,7,0.040
2,Random Forest,Fare,0.010,0.023,0%,40%,4,7,0.191
6,Random Forest,SibSp,0.008,0.012,0%,60%,5,7,0.038
3,Random Forest,Parch,0.007,0.005,0%,60%,4,7,0.035
12,Дерево,Sex,0.244,0.034,100%,100%,1,1,0.591
11,Дерево,Pclass,0.097,0.020,100%,100%,2,2,0.249
7,Дерево,Age,0.035,0.005,80%,100%,3,4,0.078


Spearman Fare–Pclass: -0.698 | SibSp–Parch: 0.422 | Fare–Sex_male: -0.254


In [12]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for ax, name in zip(axes, ["Дерево", "Random Forest"]):
    part = perm_summary[perm_summary["Модель"] == name].set_index("Ознака").loc[FEATURES]
    x = np.arange(len(FEATURES))
    ax.bar(x - 0.2, part["Impurity"], 0.4, color="#c46a5a", label="impurity (train, частка)")
    ax.bar(x + 0.2, part["PI_mean"], 0.4, yerr=part["PI_fold_std"], capsize=3, color="#3f7f93", label="permutation (Δ macro F1, ± std fold-ів)")
    ax.set_xticks(x, FEATURES)
    ax.set(title=f"{name}: impurity vs permutation importance", ylabel="важливість")
    ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIGURES / "importance.png", dpi=160, bbox_inches="tight")
plt.show()

## 9. Partial dependence для фінального ансамблю

PDP показує усереднену ймовірність класу **1 (не вижив)** для Random Forest при штучній зміні `Age` і `Fare`; риски внизу — децилі навчальних значень (де їх нема, висновок ненадійний).

In [13]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
PartialDependenceDisplay.from_estimator(rf_full, X_train, features=["Age", "Fare"], kind="average", ax=axes,
                                        line_kw={"color": "#3f7f93"})
for ax in np.ravel(axes):
    ax.set_ylabel("P(не вижив), усереднено")
fig.suptitle("Partial dependence (Random Forest, клас 1 — не вижив)")
fig.tight_layout()
fig.savefig(FIGURES / "partial_dependence.png", dpi=160, bbox_inches="tight")
plt.show()

from sklearn.inspection import partial_dependence
pd_rows = []
for feat in ["Age", "Fare"]:
    r = partial_dependence(rf_full, X_train, [feat], kind="average")
    grid_values, avg = np.asarray(r["grid_values"][0], dtype=float), r["average"][0]
    keep = ~np.isnan(grid_values)
    grid_values, avg = grid_values[keep], avg[keep]
    for q in [0.05, 0.25, 0.5, 0.75, 0.95]:
        v = np.nanquantile(X_train[feat], q)
        j = np.argmin(np.abs(grid_values - v))
        pd_rows.append({"Ознака": feat, "квантиль": q, "значення": grid_values[j], "PD P(не вижив)": avg[j]})
display(pd.DataFrame(pd_rows).style.format({"значення": "{:.1f}", "PD P(не вижив)": "{:.3f}"}))

,Ознака,квантиль,значення,PD P(не вижив)
0,Age,0.050000,5.0,0.358
1,Age,0.250000,21.0,0.617
2,Age,0.500000,29.0,0.622
3,Age,0.750000,39.0,0.652
4,Age,0.950000,58.0,0.704
5,Fare,0.050000,7.1,0.655
6,Fare,0.250000,8.3,0.656
7,Fare,0.500000,14.0,0.644
8,Fare,0.750000,31.1,0.597
9,Fare,0.950000,116.5,0.530


## 10. Розбіжності прогнозів (спільна 5-fold схема)

In [14]:
cv5 = make_cv()
oof = pd.DataFrame({"y_true": y_train}, index=X_train.index)
for name, est in fixed.items():
    oof[name] = cross_val_predict(est, X_train, y_train, cv=cv5, n_jobs=2)
oof["Tree proba"] = cross_val_predict(best_tree, X_train, y_train, cv=cv5, method="predict_proba", n_jobs=2)[:, 1]
oof["RF proba"] = cross_val_predict(models["Random Forest"], X_train, y_train, cv=cv5, method="predict_proba", n_jobs=2)[:, 1]
t_ok = oof["Контрольоване дерево"] == oof["y_true"]
rf_ok = oof["Random Forest"] == oof["y_true"]
hgb_ok = oof["HistGradientBoosting"] == oof["y_true"]
categories = pd.Series({
    "Дерево помилилось, обидва ансамблі праві": int((~t_ok & rf_ok & hgb_ok).sum()),
    "Дерево праве, хоча б один ансамбль помилився": int((t_ok & (~rf_ok | ~hgb_ok)).sum()),
    "Ансамблі дали різні класи": int((oof["Random Forest"] != oof["HistGradientBoosting"]).sum()),
    "Усі три моделі помилились": int((~t_ok & ~rf_ok & ~hgb_ok).sum()),
})
display(categories.to_frame("Об'єктів"))
pair = pd.DataFrame(index=list(fixed), columns=list(fixed), dtype=float)
for a in fixed:
    for b in fixed:
        pair.loc[a, b] = (oof[a] != oof[b]).mean()
display(pair.style.format("{:.1%}"))
dis = oof[(oof["Контрольоване дерево"] != oof["Random Forest"]) | (oof["Random Forest"] != oof["HistGradientBoosting"])].join(X_train)
dis.to_csv(RESULTS / "oof_disagreements.csv")
examples = pd.concat([dis[~t_ok.loc[dis.index] & rf_ok.loc[dis.index]].head(4), dis[t_ok.loc[dis.index] & ~rf_ok.loc[dis.index]].head(3),
                      dis[dis["Random Forest"] != dis["HistGradientBoosting"]].head(5)])
examples = examples[~examples.index.duplicated()].head(10)
display(examples[["y_true", "Контрольоване дерево", "Random Forest", "HistGradientBoosting", "Tree proba", "RF proba"] + FEATURES].style.format(
    {"Tree proba": "{:.2f}", "RF proba": "{:.2f}", "Age": "{:.0f}", "Fare": "{:.1f}"}))
print("Розбіжності за групами (стать, клас):")
display(dis.groupby(["Sex", "Pclass"]).size().to_frame("розбіжностей").join(X_train.groupby(["Sex", "Pclass"]).size().to_frame("усього")))

,Об'єктів
"Дерево помилилось, обидва ансамблі праві",34
"Дерево праве, хоча б один ансамбль помилився",46
Ансамблі дали різні класи,36
Усі три моделі помилились,89


,Контрольоване дерево,Random Forest,HistGradientBoosting
Контрольоване дерево,0.0%,9.6%,11.8%
Random Forest,9.6%,0.0%,5.1%
HistGradientBoosting,11.8%,5.1%,0.0%


,y_true,Контрольоване дерево,Random Forest,HistGradientBoosting,Tree proba,RF proba,Age,SibSp,Parch,Fare,Pclass,Sex,Embarked
PassengerId,,,,,,,,,,,,,
646,0,1,0,1,0.66,0.47,48,1,0,76.7,1,male,C
808,1,0,1,0,0.29,0.58,18,0,0,7.8,3,female,S
17,1,0,1,1,0.00,0.57,2,4,1,29.1,3,male,Q
730,1,0,1,1,0.46,0.72,25,1,0,7.9,3,female,S
290,0,0,1,1,0.46,0.63,22,0,0,7.8,3,female,Q
138,1,1,0,0,0.61,0.27,37,1,0,53.1,1,male,S
637,1,1,0,0,0.89,0.45,32,0,0,7.9,3,male,S
378,1,1,1,0,0.60,0.52,27,0,2,211.5,1,male,C
568,1,1,1,0,0.68,0.57,29,0,4,21.1,3,female,S


Розбіжності за групами (стать, клас):


розбіжностей  усього
Sex    Pclass                      
female 3                 43     113
male   1                 40     102
       2                  1      90
       3                 10     272

In [15]:
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(pair.astype(float) * 100, annot=True, fmt=".1f", cmap="Oranges", ax=ax, cbar_kws={"label": "% об'єктів"})
ax.set(title="Попарна розбіжність OOF-прогнозів, %")
fig.tight_layout()
fig.savefig(FIGURES / "pairwise_disagreement.png", dpi=160, bbox_inches="tight")
plt.show()

## 11. Інженерні характеристики

In [16]:
X_bench = X_train.sample(1000, replace=True, random_state=RANDOM_STATE)
eng_rows = []
for name, est in fixed.items():
    fits = []
    for _ in range(3):
        s = time.perf_counter(); fitted = clone(est).fit(X_train, y_train); fits.append(time.perf_counter() - s)
    fitted.predict(X_bench)
    preds = []
    for _ in range(7):
        s = time.perf_counter(); fitted.predict(X_bench); preds.append(time.perf_counter() - s)
    model = fitted.named_steps["model"]
    if hasattr(model, "tree_"):
        nodes, trees = model.tree_.node_count, 1
    elif hasattr(model, "estimators_"):
        nodes, trees = sum(e.tree_.node_count for e in model.estimators_), len(model.estimators_)
    else:
        trees = model.n_iter_
        nodes = sum(len(p.nodes) for it in model._predictors for p in it)
    eng_rows.append({"Модель": name, "Fit, мс (медіана 3)": np.median(fits) * 1000, "Predict 1000, мс (медіана 7)": np.median(preds) * 1000,
                     "Розмір, КБ": len(pickle.dumps(fitted)) / 1024, "Дерев/ітерацій": trees, "Вузлів": nodes,
                     "Пояснення окремого прогнозу": {"Контрольоване дерево": "шлях у дереві (decision_path)",
                                                    "Random Forest": "лише сурогатні: PI, PDP, частка голосів дерев",
                                                    "HistGradientBoosting": "лише сурогатні: PI, PDP"}[name]})
engineering = pd.DataFrame(eng_rows)
engineering.to_csv(RESULTS / "engineering.csv", index=False)
display(engineering.style.format({"Fit, мс (медіана 3)": "{:.1f}", "Predict 1000, мс (медіана 7)": "{:.2f}", "Розмір, КБ": "{:.1f}"}))
print(f"Python {platform.python_version()}, scikit-learn {sklearn.__version__}, NumPy {np.__version__}, pandas {pd.__version__}; "
      f"CPU: {platform.machine()}, {__import__('os').cpu_count()} логічних ядра")

,Модель,"Fit, мс (медіана 3)","Predict 1000, мс (медіана 7)","Розмір, КБ",Дерев/ітерацій,Вузлів,Пояснення окремого прогнозу
0,Контрольоване дерево,9.8,4.47,4.5,1,15,шлях у дереві (decision_path)
1,Random Forest,444.4,33.90,4164.2,300,51874,"лише сурогатні: PI, PDP, частка голосів дерев"
2,HistGradientBoosting,199.8,15.22,352.3,200,5792,"лише сурогатні: PI, PDP"


Python 3.13.16, scikit-learn 1.7.1, NumPy 2.3.2, pandas 2.3.2; CPU: x86_64, 2 логічних ядра


## 12. Вибір моделі (до test) і фінальна перевірка

In [17]:
q = quality.set_index("Модель")
s = stability.set_index("Модель")
tree_f1, tree_sd = q.loc["Контрольоване дерево (nested)", "CV F1"], q.loc["Контрольоване дерево (nested)", "CV std"]
decision_rows = []
for ens in ["Random Forest", "HistGradientBoosting"]:
    decision_rows.append({"Ансамбль": ens, "ΔCV F1 vs дерево": q.loc[ens, "CV F1"] - tree_f1, "SD дерева": tree_sd,
                          "Розбіжність дерево / ансамбль": f"{s.loc['Контрольоване дерево', 'Розбіжність з консенсусом']:.2%} / "
                                                           f"{s.loc[ens, 'Розбіжність з консенсусом']:.2%}",
                          "Умова якості": q.loc[ens, "CV F1"] - tree_f1 > tree_sd,
                          "Умова стійкості": s.loc[ens, "Розбіжність з консенсусом"] < s.loc["Контрольоване дерево", "Розбіжність з консенсусом"]})
decision = pd.DataFrame(decision_rows)
display(decision.style.format({"ΔCV F1 vs дерево": "{:+.3f}", "SD дерева": "{:.3f}"}))
passed = decision[decision["Умова якості"] & decision["Умова стійкості"]]
if len(passed):
    FINAL = passed.sort_values("ΔCV F1 vs дерево", ascending=False).iloc[0]["Ансамбль"]
else:
    FINAL = "Контрольоване дерево"
print("Фінальна модель за правилом протоколу:", FINAL)

,Ансамбль,ΔCV F1 vs дерево,SD дерева,Розбіжність дерево / ансамбль,Умова якості,Умова стійкості
0,Random Forest,+0.014,0.033,2.23% / 3.57%,False,False
1,HistGradientBoosting,+0.011,0.033,2.23% / 4.47%,False,False


Фінальна модель за правилом протоколу: Контрольоване дерево


In [18]:
final_estimator = fixed[FINAL]
final_params = {k.replace("model__", ""): v for k, v in final_estimator.get_params().items()
                if k.startswith("model__") and not callable(v)}
config = {"model": FINAL, "params": {k: (v if isinstance(v, (int, float, str, bool, type(None))) else str(v)) for k, v in final_params.items()},
          "preprocessing": "median imputation + missing indicator (numeric), most_frequent + one-hot (categorical), no scaling",
          "features": FEATURES, "target": "AtRisk = 1 - Survived", "metric": "f1_macro",
          "split": {"test_size": 0.2, "stratify": True, "random_state": RANDOM_STATE},
          "outer_cv": "RepeatedStratifiedKFold(5x4, random_state=42)",
          "selection_rule": "ensemble only if CV macro F1 gain > tree SD and OOF instability lower; otherwise controlled tree",
          "evidence": {"cv_macro_f1": round(float(q.loc[FINAL if FINAL != "Контрольоване дерево" else "Контрольоване дерево (nested)", "CV F1"]), 4),
                       "oof_instability": round(float(s.loc[FINAL, "Розбіжність з консенсусом"]), 4)},
          "controlled_tree_reference": {k.replace("model__", ""): v for k, v in tree_search.best_params_.items()}}
(CONFIGS / "final_model.json").write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(config, ensure_ascii=False, indent=2))

{
  "model": "Контрольоване дерево",
  "params": {
    "ccp_alpha": 0.0,
    "class_weight": null,
    "criterion": "gini",
    "max_depth": 3,
    "max_features": null,
    "max_leaf_nodes": null,
    "min_impurity_decrease": 0.0,
    "min_samples_leaf": 2,
    "min_samples_split": 2,
    "min_weight_fraction_leaf": 0.0,
    "monotonic_cst": null,
    "random_state": 42,
    "splitter": "best"
  },
  "preprocessing": "median imputation + missing indicator (numeric), most_frequent + one-hot (categorical), no scaling",
  "features": [
    "Age",
    "SibSp",
    "Parch",
    "Fare",
    "Pclass",
    "Sex",
    "Embarked"
  ],
  "target": "AtRisk = 1 - Survived",
  "metric": "f1_macro",
  "split": {
    "test_size": 0.2,
    "stratify": true,
    "random_state": 42
  },
  "outer_cv": "RepeatedStratifiedKFold(5x4, random_state=42)",
  "selection_rule": "ensemble only if CV macro F1 gain > tree SD and OOF instability lower; otherwise controlled tree",
  "evidence": {
    "cv_macro_f1": 0.

In [19]:
final_model = clone(final_estimator).fit(X_train, y_train)
y_pred = final_model.predict(X_test)
cm = confusion_matrix(y_test, y_pred)
test_metrics = pd.DataFrame([{"Модель": FINAL, "Test macro F1": f1_score(y_test, y_pred, average="macro"),
                              "Test Accuracy": accuracy_score(y_test, y_pred), "TN": cm[0, 0], "FP": cm[0, 1], "FN": cm[1, 0], "TP": cm[1, 1],
                              "CV macro F1": config["evidence"]["cv_macro_f1"]}])
test_metrics.to_csv(RESULTS / "test_metrics.csv", index=False)
display(test_metrics.style.format({"Test macro F1": "{:.3f}", "Test Accuracy": "{:.3f}", "CV macro F1": "{:.3f}"}))
fig, ax = plt.subplots(figsize=(5.5, 4.5))
ConfusionMatrixDisplay(cm, display_labels=["Вижив (0)", "Не вижив (1)"]).plot(ax=ax, values_format="d", cmap="Blues", colorbar=False)
ax.set(title=f"{FINAL}: confusion matrix (test)", xlabel="Прогноз", ylabel="Факт")
fig.tight_layout()
fig.savefig(FIGURES / "final_confusion_matrix.png", dpi=160, bbox_inches="tight")
plt.show()
test_errors = X_test[y_pred != y_test].assign(y_true=y_test[y_pred != y_test],
                                              proba=final_model.predict_proba(X_test)[y_pred != y_test, 1])
test_errors.to_csv(RESULTS / "test_errors.csv")
display(test_errors.groupby(["Sex", "Pclass", "y_true"]).size().to_frame("помилок"))

,Модель,Test macro F1,Test Accuracy,TN,FP,FN,TP,CV macro F1
0,Контрольоване дерево,0.800,0.816,48,21,12,98,0.789


помилок
Sex    Pclass y_true         
female 1      1             1
       2      1             4
       3      0             2
              1             5
male   1      0             6
       2      0             2
       3      0            11
              1             2